In [ ]:
import tensorflow as tf

import os
import pathlib
import time
import datetime

from pathlib import Path
from matplotlib import pyplot as plt
from IPython import display
import numpy as np

### Definição do modelo

In [ ]:
OUTPUT_CHANNELS = 3

# Paths:
PATH_INPUT = 'sprites-processed-countours'
PATH_REAL = 'sprites-processed'

# Canais de cada lado do par. Os contornos sao PNG em tons de cinza (modo L) e os
# alvos sao RGB; passar isso explicitamente ao decode_png evita depender do que o
# arquivo declara no cabecalho.
INPUT_CHANNELS = 1
REAL_CHANNELS = 3

# Tamanho de cada imagem:
IMG_WIDTH = 256
IMG_HEIGHT = 256

# The batch size of 1 produced better results for the U-Net in the original pix2pix experiment
BATCH_SIZE = 1

# Split treino/teste: tamanho e semente fixos, aplicados sobre a lista de arquivos.
TEST_SIZE = 1000
SPLIT_SEED = 1337

# Treinamento:
LAMBDA = 100

In [ ]:
def downsample(filters, size, apply_batchnorm=True):
  initializer = tf.random_normal_initializer(0., 0.02)

  result = tf.keras.Sequential()
  result.add(tf.keras.layers.Conv2D(filters, size, strides=2, padding='same', kernel_initializer=initializer, use_bias=False))

  if apply_batchnorm:
    result.add(tf.keras.layers.BatchNormalization())

  result.add(tf.keras.layers.LeakyReLU())

  return result

def upsample(filters, size, apply_dropout=False):
  initializer = tf.random_normal_initializer(0., 0.02)

  result = tf.keras.Sequential()
  result.add(tf.keras.layers.Conv2DTranspose(filters, size, strides=2, padding='same', kernel_initializer=initializer, use_bias=False))
  result.add(tf.keras.layers.BatchNormalization())

  if apply_dropout:
      result.add(tf.keras.layers.Dropout(0.5))

  result.add(tf.keras.layers.ReLU())

  return result

def Generator():
  inputs = tf.keras.layers.Input(shape=[256, 256, 1])

  down_stack = [
    downsample(64, 4, apply_batchnorm=False),  # (batch_size, 128, 128, 64)
    downsample(128, 4),  # (batch_size, 64, 64, 128)
    downsample(256, 4),  # (batch_size, 32, 32, 256)
    downsample(512, 4),  # (batch_size, 16, 16, 512)
    downsample(512, 4),  # (batch_size, 8, 8, 512)
    downsample(512, 4),  # (batch_size, 4, 4, 512)
    downsample(512, 4),  # (batch_size, 2, 2, 512)
    downsample(512, 4),  # (batch_size, 1, 1, 512)
  ]

  up_stack = [
    upsample(512, 4, apply_dropout=True),  # (batch_size, 2, 2, 1024)
    upsample(512, 4, apply_dropout=True),  # (batch_size, 4, 4, 1024)
    upsample(512, 4, apply_dropout=True),  # (batch_size, 8, 8, 1024)
    upsample(512, 4),  # (batch_size, 16, 16, 1024)
    upsample(256, 4),  # (batch_size, 32, 32, 512)
    upsample(128, 4),  # (batch_size, 64, 64, 256)
    upsample(64, 4),  # (batch_size, 128, 128, 128)
  ]

  initializer = tf.random_normal_initializer(0., 0.02)
  last = tf.keras.layers.Conv2DTranspose(OUTPUT_CHANNELS, 4, strides=2, padding='same', kernel_initializer=initializer, activation='tanh')  # (batch_size, 256, 256, 3)

  x = inputs

  # Downsampling through the model
  skips = []
  for down in down_stack:
    x = down(x)
    skips.append(x)

  skips = reversed(skips[:-1])

  # Upsampling and establishing the skip connections
  for up, skip in zip(up_stack, skips):
    x = up(x)
    x = tf.keras.layers.Concatenate()([x, skip])

  x = last(x)

  return tf.keras.Model(inputs=inputs, outputs=x)

### Leitura da entrada

In [ ]:
def load(image_file, channels):
    image = tf.io.read_file(image_file)
    image = tf.io.decode_png(image, channels=channels)
    return tf.cast(image, tf.float32)

def resize(image, height, width):
    return tf.image.resize(image, [height, width], method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)

def random_crop(image, seed):
    return tf.image.stateless_random_crop(image, seed=seed, size=[IMG_HEIGHT, IMG_WIDTH, tf.shape(image)[2]])

def normalize(image):
    return (image / 127.5) - 1

def random_jitter(image, seed):
  # Resizing to 286x286 and random cropping back to 256x256
  image = resize(image, 286, 286)
  image = random_crop(image, seed)
  image = tf.image.stateless_random_flip_left_right(image, seed)

  return image

def gen_seed_pair():
   # Precisa ser um op do TF. Esta funcao e tracada para dentro do grafo do tf.data,
   # e um sorteio feito em Python (np.random.randint) seria avaliado uma unica vez,
   # durante o tracing, virando constante: o dataset inteiro receberia o mesmo crop
   # e o mesmo flip. tf.random.uniform vira um no do grafo e sorteia por elemento.
   return tf.random.uniform([2], maxval=2 ** 31 - 1, dtype=tf.int32)

def load_image_pair(image_file_input, image_file_real):
  # Treino: a mesma semente nos dois lados garante crop e flip identicos no par.
  seed = gen_seed_pair()
  image_input = load(image_file_input, INPUT_CHANNELS)
  image_real = load(image_file_real, REAL_CHANNELS)
  image_input, image_real = random_jitter(image_input, seed), random_jitter(image_real, seed)

  return normalize(image_input), normalize(image_real)

def load_image_pair_eval(image_file_input, image_file_real):
  # Teste: sem jitter, para que a avaliacao seja a mesma entre rodadas.
  image_input = load(image_file_input, INPUT_CHANNELS)
  image_real = load(image_file_real, REAL_CHANNELS)
  image_input = resize(image_input, IMG_HEIGHT, IMG_WIDTH)
  image_real = resize(image_real, IMG_HEIGHT, IMG_WIDTH)

  return normalize(image_input), normalize(image_real)

#### Teste de leitura

In [ ]:
inp = load(str(f'{PATH_INPUT}/gen9/bulbasaur.png'), INPUT_CHANNELS)
re = load(str(f'{PATH_REAL}/gen9/bulbasaur.png'), REAL_CHANNELS)
# Casting to int for matplotlib to display the images
f, axarr = plt.subplots(1, 2)
axarr[0].imshow(inp / 255.0, cmap='gray')
axarr[1].imshow(re / 255.0)

In [ ]:
seed = gen_seed_pair()
rj_inp = random_jitter(inp, seed)
rj_re = random_jitter(re, seed)
f, axarr = plt.subplots(1, 2)
axarr[0].imshow(rj_inp / 255.0, cmap='gray')
axarr[1].imshow(rj_re / 255.0)

### Constroi pipeline

Split fixo sobre a lista de arquivos, antes de qualquer embaralhamento.

In [ ]:
# Lista ordenada e deterministica dos dois lados do par:
input_paths = sorted(Path(PATH_INPUT).glob('*/*.png'))
real_paths = sorted(Path(PATH_REAL).glob('*/*.png'))

# Confere o pareamento antes de treinar: geracao e nome do arquivo tem que bater.
assert len(input_paths) == len(real_paths), (len(input_paths), len(real_paths))
for p_in, p_re in zip(input_paths, real_paths):
    assert (p_in.parent.name, p_in.name) == (p_re.parent.name, p_re.name), (p_in, p_re)

input_files = [str(p) for p in input_paths]
real_files = [str(p) for p in real_paths]
print(f'{len(input_files)} pares: {PATH_INPUT} -> {PATH_REAL}')

# Split fixo, sorteado UMA vez sobre a lista de arquivos.
# O split anterior era take(1000)/skip(1000) aplicado depois de um .shuffle() com
# reshuffle_each_iteration=True (o padrao). Como take/skip sao views preguicosas,
# cada passada pelo dataset reembaralhava e recortava um conjunto diferente: o
# teste vazava para o treino e as imagens da avaliacao nao eram dados nao vistos.
perm = np.random.default_rng(SPLIT_SEED).permutation(len(input_files))
test_idx, train_idx = perm[:TEST_SIZE], perm[TEST_SIZE:]
assert set(train_idx.tolist()).isdisjoint(test_idx.tolist())

def make_dataset(indices, map_fn, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((
        [input_files[i] for i in indices],
        [real_files[i] for i in indices],
    ))
    if shuffle:
        # Embaralha a lista de caminhos, nao as imagens decodificadas: o buffer
        # cobre o dataset inteiro de graca, em vez dos 2000 parciais de antes.
        ds = ds.shuffle(len(indices), reshuffle_each_iteration=True)
    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_dataset = make_dataset(train_idx, load_image_pair, shuffle=True)
test_dataset = make_dataset(test_idx, load_image_pair_eval, shuffle=False)

print(f'treino: {len(train_idx)} pares | teste: {len(test_idx)} pares | disjuntos, seed={SPLIT_SEED}')

# OBS: o split ainda e por imagem, nao por pokemon. O mesmo pokemon aparece em ate
# 9 geracoes com arte parecida, entao parte do conjunto de teste tem um "primo" no
# treino. Corrigir isso exige agrupar por p.name antes de permutar.

### Treinamento

#### Teste

In [ ]:
generator = Generator()
gen_output = generator(inp[tf.newaxis, ...], training=False)
plt.imshow(gen_output[0, ...])

In [ ]:
loss_object = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(disc_generated_output, gen_output, target):
  gan_loss = loss_object(tf.ones_like(disc_generated_output), disc_generated_output)
  # Mean absolute error
  l1_loss = tf.reduce_mean(tf.abs(target - gen_output))
  total_gen_loss = gan_loss + (LAMBDA * l1_loss)
  return total_gen_loss, gan_loss, l1_loss

#### Discriminador

In [ ]:
def Discriminator():
    initializer = tf.random_normal_initializer(0., 0.02)

    inp = tf.keras.layers.Input(shape=[256, 256, 1], name='input_image')
    tar = tf.keras.layers.Input(shape=[256, 256, 3], name='target_image')

    x = tf.keras.layers.concatenate([inp, tar])  # (batch_size, 256, 256, channels*2)

    down1 = downsample(64, 4, False)(x)  # (batch_size, 128, 128, 64)
    down2 = downsample(128, 4)(down1)  # (batch_size, 64, 64, 128)
    down3 = downsample(256, 4)(down2)  # (batch_size, 32, 32, 256)

    zero_pad1 = tf.keras.layers.ZeroPadding2D()(down3)  # (batch_size, 34, 34, 256)
    conv = tf.keras.layers.Conv2D(512, 4, strides=1, kernel_initializer=initializer, use_bias=False)(zero_pad1)  # (batch_size, 31, 31, 512)

    batchnorm1 = tf.keras.layers.BatchNormalization()(conv)
    leaky_relu = tf.keras.layers.LeakyReLU()(batchnorm1)
    zero_pad2 = tf.keras.layers.ZeroPadding2D()(leaky_relu)  # (batch_size, 33, 33, 512)

    last = tf.keras.layers.Conv2D(1, 4, strides=1, kernel_initializer=initializer)(zero_pad2)  # (batch_size, 30, 30, 1)

    return tf.keras.Model(inputs=[inp, tar], outputs=last)

def discriminator_loss(disc_real_output, disc_generated_output):
    real_loss = loss_object(tf.ones_like(disc_real_output), disc_real_output)
    generated_loss = loss_object(tf.zeros_like(disc_generated_output), disc_generated_output)
    total_disc_loss = real_loss + generated_loss
    return total_disc_loss

In [ ]:
discriminator = Discriminator()
disc_out = discriminator([inp[tf.newaxis, ...], gen_output], training=False)
print(disc_out.shape)
plt.imshow(disc_out[0, ..., -1], vmin=-20, vmax=20, cmap='RdBu_r')
plt.colorbar()

### Discriminador

In [ ]:
def discriminator_loss(disc_real_output, disc_generated_output):
  real_loss = loss_object(tf.ones_like(disc_real_output), disc_real_output)

  generated_loss = loss_object(tf.zeros_like(disc_generated_output), disc_generated_output)

  total_disc_loss = real_loss + generated_loss

  return total_disc_loss

### Define otimizador ADAM e checkpoint

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

In [ ]:
checkpoint_dir = f'./{PATH_INPUT}_training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer, discriminator_optimizer=discriminator_optimizer, generator=generator, discriminator=discriminator)

In [ ]:
def generate_images(model, test_input, tar):
  prediction = model(test_input, training=True)
  plt.figure(figsize=(15, 15))

  display_list = [test_input[0], tar[0], prediction[0]]
  title = ['Input Image', 'Ground Truth', 'Predicted Image']

  for i in range(3):
    plt.subplot(1, 3, i+1)
    plt.title(title[i])
    # Getting the pixel values in the [0, 1] range to plot.
    plt.imshow(display_list[i] * 0.5 + 0.5, cmap='gray')
    plt.axis('off')
  plt.show()

In [ ]:
for example_input, example_target in train_dataset.take(1):
  generate_images(generator, example_input, example_target)

### Treinamento

In [ ]:
log_dir = "logs/"
run_name = f'{PATH_INPUT}-{datetime.datetime.now().strftime("%Y%m%d-%H%M%S")}'
summary_writer = tf.summary.create_file_writer(log_dir + "fit/" + run_name)
print(f'logs desta rodada: {log_dir}fit/{run_name}')

In [ ]:
@tf.function
def train_step(input_image, target, step):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_output = generator(input_image, training=True)

        disc_real_output = discriminator([input_image, target], training=True)
        disc_generated_output = discriminator([input_image, gen_output], training=True)

        gen_total_loss, gen_gan_loss, gen_l1_loss = generator_loss(disc_generated_output, gen_output, target)
        disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

        generator_gradients = gen_tape.gradient(gen_total_loss,generator.trainable_variables)
        discriminator_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

        generator_optimizer.apply_gradients(zip(generator_gradients, generator.trainable_variables))
        discriminator_optimizer.apply_gradients(zip(discriminator_gradients, discriminator.trainable_variables))

    with summary_writer.as_default():
        tf.summary.scalar('gen_total_loss', gen_total_loss, step=step//1000)
        tf.summary.scalar('gen_gan_loss', gen_gan_loss, step=step//1000)
        tf.summary.scalar('gen_l1_loss', gen_l1_loss, step=step//1000)
        tf.summary.scalar('disc_loss', disc_loss, step=step//1000)

In [ ]:
def fit(train_ds, test_ds, steps):
    example_input, example_target = next(iter(test_ds.take(1)))
    start = time.time()

    for step, (input_image, target) in train_ds.repeat().take(steps).enumerate():
        if (step) % 1000 == 0:
            display.clear_output(wait=True)
            if step != 0:
                print(f'Time taken for 1000 steps: {time.time()-start:.2f} sec\n')
            start = time.time()
            generate_images(generator, example_input, example_target)
            print(f"Step: {step//1000}k")
        train_step(input_image, target, step)
        # Training step
        if (step+1) % 10 == 0:
            print('.', end='', flush=True)
        # Save (checkpoint) the model every 5k steps
        if (step + 1) % 5000 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)

In [ ]:
fit(train_dataset, test_dataset, steps=40000)

In [ ]:
checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

In [ ]:
# Run the trained model on a few examples from the test set
for inp, tar in test_dataset.take(12):
    generate_images(generator, inp, tar)